In [8]:
import json
import os
import pandas as pd
import numpy as np
import cv2

DATA_DIR = '../input/competitions/cassava-leaf-disease-classification/'

with open(os.path.join(DATA_DIR, 'label_num_to_disease_map.json')) as f:
    disease_map = json.load(f)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# 숫자로 된 label을 실제 질병 이름으로 바꾼 열을 하나 더 추가
df['disease_name'] = df['label'].astype(str).map(disease_map)

In [9]:
from sklearn.model_selection import train_test_split

#클래스 불균형이 발생했으므로 label의 개수를 동일하게 split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"학습용 데이터 개수: {len(train_df)}장")
print(f"테스트용 데이터 개수: {len(test_df)}장")

학습용 데이터 개수: 17117장
테스트용 데이터 개수: 4280장


In [10]:
import torch
from torch.utils.data import Dataset

class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform 
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_name = self.df.loc[index, 'image_id']
        label = self.df.loc[index, 'label']

        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
        
        if self.transform is not None:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [12]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1. / class_counts

sample_weights = [class_weights[label] for label in train_df['label']]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights))

In [13]:
from torch.utils.data import DataLoader

train_dataset = CassavaDataset(df=train_df, image_dir=DATA_DIR+'train_images', transform=transform)
test_dataset = CassavaDataset(df=test_df, image_dir=DATA_DIR+'train_images', transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=0   
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

In [14]:
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 5)
)
model = model.to(device)

In [15]:
# 출력층 개수 변경

n_features = model.fc.in_features
model.fc = nn.Linear(n_features, 5)

In [16]:
# 모델 GPU로 전송

if torch.cuda.is_available():
    print("yes")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

yes


In [17]:
#손실함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

In [ ]:
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

epochs = 20

scaler = GradScaler(device='cuda')
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

best_val_acc = 0
patience = 7
counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        #AMP
        with autocast(device_type='cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    epoch_loss = running_loss / len(train_loader)
    train_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= len(test_loader)
    val_acc = val_correct / val_total

    #scheduler
    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    #Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early Stopping 발동")
        break

print("학습 완료")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_labels, all_preds)
print(f"Test 정확도: {final_accuracy * 100:.2f}%")

print("\n Classification Report")

disease_names = ['CBB (0)', 'CBSD (1)', 'CGM (2)', 'CMD (3)', 'Healthy (4)']
print(classification_report(all_labels, all_preds, target_names=disease_names))